# Desarrollo MLP: predicción y clustering en validation

Este notebook evalúa el MLP primario `32→64→64→32` sobre datos sintéticos nuevos. Selecciona checkpoints sólo mediante predicción y no-colapso; luego mide clustering en validation. **No construye ni consulta test.**

## Protocolo congelado

La configuración usa `base_seed=1`, 64/32 masters train/validation por régimen, seeds 10–14 y 20 épocas. La pureza de cada modelo es la media de 20 `random_state` de K-means con `n_init=20`. El gate exige media global ≥60%, peor seed ≥55%, CV entre seeds ≤10% y sd intra-seed ≤3 puntos. `65.48%` se muestra sólo como referencia descriptiva del paper.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import (
    PaperCheckpointReplayConfig,
    load_paper_mlp_clustering_development_config,
)
from koopman_jepa.paper_data import PAPER_REGIME_NAMES, PaperRegimeDataset, generate_paper_master
from koopman_jepa.paper_evaluation import (
    evaluate_paper_mlp_clustering_gate,
    evaluate_paper_mlp_seed_clustering,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    evaluate_scale_invariant_seed_stability_gate,
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    summarize_seed_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_mlp_clustering_development.yaml"
config = load_paper_mlp_clustering_development_config(config_path)
replay_config = PaperCheckpointReplayConfig(metric_absolute_tolerance=1e-8)
torch.use_deterministic_algorithms(True)
print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {validation_dataset.sample_key(index) for index in range(len(validation_dataset))}
assert config.data.base_seed == 1
assert len(train_dataset) == 64 * len(PAPER_REGIME_NAMES) == 1152
assert len(validation_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation nuevos: {len(train_dataset)}/{len(validation_dataset)}")
print("Test no fue instanciado.")

In [ ]:
models, replay_results, summaries, histories, times = {}, {}, [], {}, {}
for seed in config.sweep.seeds:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    run_config = replace(config.train, seed=seed)
    model = PaperTemporalJEPA(config.model)
    started = time.perf_counter()
    checkpoint_run = run_paper_train_validation_with_checkpoint(
        model, train_dataset, validation_dataset, run_config, config.checkpoint_gate
    )
    if checkpoint_run.selection is None:
        times[seed] = time.perf_counter() - started
        print(f"seed={seed} checkpoint=NONE time={times[seed]:.2f}s")
        continue
    replay = verify_paper_checkpoint_replay(
        model, checkpoint_run, validation_dataset, run_config, replay_config,
        expected_epoch=checkpoint_run.selection.epoch,
    )
    times[seed] = time.perf_counter() - started
    models[seed], replay_results[seed] = model, replay
    summaries.append(summarize_seed_checkpoint(seed, checkpoint_run.selection))
    histories[seed] = checkpoint_run.history
    print(
        f"seed={seed} checkpoint={checkpoint_run.selection.epoch} "
        f"val/base={checkpoint_run.selection.validation_loss_ratio:.3f} "
        f"gap={checkpoint_run.selection.validation_train_loss_ratio:.3f} "
        f"rank={checkpoint_run.selection.validation_effective_rank:.2f} "
        f"replay={'PASS' if replay.passed else 'FAIL'} time={times[seed]:.2f}s"
    )
replay_passed = set(replay_results) == set(config.sweep.seeds) and all(row.passed for row in replay_results.values())
stability_gate = evaluate_scale_invariant_seed_stability_gate(summaries, config.sweep, config.stability_gate)
assert replay_passed
print(f"Replay: PASS; tiempo total={sum(times.values()):.2f}s")
print("Gate predictivo:", json.dumps(asdict(stability_gate), indent=2))
print("Test sigue sin instanciar.")

In [ ]:
@torch.no_grad()
def collect_validation_embeddings(model, run_config):
    device = torch.device(run_config.device)
    model.to(device).eval()
    loader = make_paper_loader(validation_dataset, run_config, shuffle=False)
    embedding_chunks, label_chunks = [], []
    for context, _, labels in loader:
        embedding_chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
        label_chunks.append(labels.numpy())
    return np.concatenate(embedding_chunks), np.concatenate(label_chunks)

clustering_metrics, validation_embeddings_by_seed = [], {}
validation_labels = None
for seed in config.sweep.seeds:
    embeddings, labels = collect_validation_embeddings(models[seed], replace(config.train, seed=seed))
    validation_labels = labels if validation_labels is None else validation_labels
    assert np.array_equal(labels, validation_labels)
    validation_embeddings_by_seed[seed] = embeddings
    clustering_metrics.append(evaluate_paper_mlp_seed_clustering(embeddings, labels, seed, config.clustering))
clustering_gate = evaluate_paper_mlp_clustering_gate(clustering_metrics, config.sweep, config.clustering_gate)
development_passed = replay_passed and stability_gate.passed and clustering_gate.passed
print("Gate de clustering:", json.dumps(asdict(clustering_gate), indent=2))
print(f"Gate conjunto: {'PASS' if development_passed else 'FAIL'}")
print("Test no fue instanciado ni consultado.")

In [ ]:
summary_by_seed = {row.seed: row for row in summaries}
clustering_by_seed = {row.seed: row for row in clustering_metrics}
print("seed epoch val/base gap rank purity_mean purity_sd purity_min purity_max matched")
for seed in config.sweep.seeds:
    summary, cluster = summary_by_seed[seed], clustering_by_seed[seed]
    print(
        f"{seed:>4d} {summary.checkpoint_epoch:>5d} {summary.validation_loss_ratio:>8.3f} "
        f"{summary.validation_train_loss_ratio:>5.3f} {summary.validation_effective_rank:>5.2f} "
        f"{cluster.mean_purity:>11.2%} {cluster.purity_std:>9.2%} "
        f"{cluster.minimum_purity:>10.2%} {cluster.maximum_purity:>10.2%} "
        f"{cluster.mean_matched_accuracy:>7.2%}"
    )

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
colors = plt.cm.tab10(np.linspace(0.0, 1.0, len(config.sweep.seeds)))
for color, seed in zip(colors, config.sweep.seeds, strict=True):
    history = histories[seed]
    epochs = np.array([row.epoch for row in history])
    validation_loss = np.array([row.validation_loss for row in history])
    train_loss = np.array([row.train_loss for row in history])
    ranks = np.array([row.validation_effective_rank for row in history])
    axes[0, 0].plot(epochs, validation_loss / validation_loss[0], marker="o", markersize=3, color=color, label=f"seed {seed}")
    axes[0, 1].plot(epochs, validation_loss / np.maximum(train_loss, 1e-12), marker="o", markersize=3, color=color)
    axes[0, 2].plot(epochs, ranks, marker="o", markersize=3, color=color)
axes[0, 0].axhline(config.checkpoint_gate.max_validation_loss_ratio, color="tab:red", linestyle="--")
axes[0, 0].set(title="Validation loss / baseline", xlabel="Época", ylabel="Ratio")
axes[0, 0].legend(ncol=2, fontsize=8)
axes[0, 1].axhline(config.checkpoint_gate.max_validation_train_loss_ratio, color="tab:red", linestyle="--")
axes[0, 1].set(title="Brecha validation/train", xlabel="Época", ylabel="Ratio")
axes[0, 2].axhline(config.checkpoint_gate.min_validation_effective_rank, color="tab:red", linestyle="--")
axes[0, 2].set(title="Rango efectivo validation", xlabel="Época", ylabel="Rango")
seeds = np.array(config.sweep.seeds)
means = np.array([clustering_by_seed[seed].mean_purity for seed in seeds])
stds = np.array([clustering_by_seed[seed].purity_std for seed in seeds])
matched = np.array([clustering_by_seed[seed].mean_matched_accuracy for seed in seeds])
axes[1, 0].errorbar(seeds, means, yerr=stds, fmt="o", capsize=5)
axes[1, 0].axhline(config.clustering_gate.min_worst_seed_mean_purity, color="tab:red", linestyle="--", label="mínimo por seed")
axes[1, 0].axhline(0.6548, color="tab:green", linestyle=":", label="paper (descriptivo)")
axes[1, 0].set(title="Pureza media ± sd K-means", xlabel="Seed", ylabel="Pureza")
axes[1, 0].legend(fontsize=8)
axes[1, 1].boxplot([clustering_by_seed[seed].purities for seed in seeds], tick_labels=seeds)
axes[1, 1].axhline(config.clustering_gate.min_overall_mean_purity, color="tab:red", linestyle="--")
axes[1, 1].set(title="Distribución por K-means state", xlabel="Seed modelo", ylabel="Pureza")
width = 0.36
axes[1, 2].bar(seeds - width / 2, means, width=width, label="pureza")
axes[1, 2].bar(seeds + width / 2, matched, width=width, label="matched")
axes[1, 2].set(title="Métricas de clustering", xlabel="Seed", ylabel="Score")
axes[1, 2].legend()
plt.show()

In [ ]:
status = "PASS" if development_passed else "FAIL"
criteria = {
    "replay": replay_passed,
    "estabilidad predictiva": stability_gate.passed,
    "media global": clustering_gate.overall_mean_passed,
    "peor seed": clustering_gate.worst_seed_passed,
    "variabilidad entre modelos": clustering_gate.seed_variability_passed,
    "estabilidad K-means": clustering_gate.kmeans_stability_passed,
}
failed_text = ", ".join(name for name, passed in criteria.items() if not passed) or "ninguno"
failed_seed_text = ", ".join(str(seed) for seed in clustering_gate.failed_seeds) or "ninguna"
selected_epochs = ", ".join(f"{row.seed}:{row.checkpoint_epoch}" for row in summaries)
decision = (
    "El desarrollo MLP pasa. Podemos congelar estos epochs y diseñar una evaluación held-out nueva."
    if development_passed
    else "El desarrollo MLP falla. Debemos diagnosticar train/validation sin abrir test ni retocar thresholds."
)
display(Markdown(f"""## Resultado e interpretación

- **Gate conjunto: {status}.**
- **Criterios fallidos:** {failed_text}.
- **Seeds fallidas:** {failed_seed_text}.
- **Epochs seleccionados:** `{selected_epochs}`.
- **Pureza media global:** `{clustering_gate.overall_mean_purity:.2%}` / mínimo `{config.clustering_gate.min_overall_mean_purity:.0%}`.
- **Peor seed:** `{clustering_gate.worst_seed_mean_purity:.2%}` / mínimo `{config.clustering_gate.min_worst_seed_mean_purity:.0%}`.
- **CV entre seeds:** `{clustering_gate.seed_mean_purity_coefficient_of_variation:.3f}` / máximo `{config.clustering_gate.max_seed_mean_purity_coefficient_of_variation:.2f}`.
- **Peor sd K-means:** `{clustering_gate.worst_within_seed_purity_std:.2%}` / máximo `{config.clustering_gate.max_within_seed_purity_std:.0%}`.

Los paneles superiores verifican predicción y rango; los inferiores separan variación entre modelos de variación de K-means. `65.48%` es sólo referencia descriptiva porque escala y split no coinciden.

### Decisión

{decision} Test no fue construido ni consultado.
"""))